#### Working - Break Invoice into Individual Invoices and save as per renaming criteria

In [7]:
import fitz  # PyMuPDF
import re
import os

def split_pdf_by_invoice(pdf_path, output_folder="ACME_individual_invoices", summary_file="invoice_summary.txt"):
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Open the PDF
    doc = fitz.open(pdf_path)
    invoice_summary = {}

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text = page.get_text()
        
        # print(f"Page {page_num + 1} text:\n{text}\n{'-'*40}")
        # Extract invoice number using regex
        match = re.search(r'INVOICE\s+(\d+)', text, re.IGNORECASE)
        if match:
            invoice_no = match.group(1)
        else:
            invoice_no = f"Unknown_{page_num+1}"

        print(f"Page {page_num + 1}: Extracted Invoice No. = {invoice_no}")

        # Save the page as a new PDF
        new_doc = fitz.open()
        new_doc.insert_pdf(doc, from_page=page_num, to_page=page_num)
        output_path = os.path.join(output_folder, f"Invoice_{invoice_no}.pdf")
        new_doc.save(output_path)
        new_doc.close()

        # Record in summary
        invoice_summary[page_num + 1] = invoice_no

    # Write summary file
    with open(summary_file, "w") as f:
        for page, invoice in invoice_summary.items():
            f.write(f"Page {page}: Invoice No. {invoice}\n")

    print(f"Done! Split into {len(invoice_summary)} files. Summary saved to {summary_file}.")

# Example usage
split_pdf_by_invoice(r"C:\Users\IlaBarshilia\ldiltd.com\M1 Global - M1 Global\Projects\FDOT Invoicing Automation\Samples for PDF Reader\JAX - June CH Billing (extracted) Done.pdf")


Page 1: Extracted Invoice No. = 168858
Page 2: Extracted Invoice No. = 168859
Page 3: Extracted Invoice No. = 168860
Page 4: Extracted Invoice No. = 168861
Page 5: Extracted Invoice No. = 168862
Page 6: Extracted Invoice No. = 168863
Page 7: Extracted Invoice No. = 168864
Page 8: Extracted Invoice No. = 168865
Page 9: Extracted Invoice No. = 168866
Page 10: Extracted Invoice No. = 168867
Page 11: Extracted Invoice No. = 168868
Page 12: Extracted Invoice No. = 168869
Page 13: Extracted Invoice No. = 168870
Page 14: Extracted Invoice No. = 168871
Page 15: Extracted Invoice No. = 168872
Page 16: Extracted Invoice No. = 168873
Page 17: Extracted Invoice No. = 168874
Page 18: Extracted Invoice No. = 168875
Page 19: Extracted Invoice No. = 168876
Page 20: Extracted Invoice No. = 168877
Page 21: Extracted Invoice No. = 168878
Page 22: Extracted Invoice No. = 168879
Page 23: Extracted Invoice No. = 168880
Page 24: Extracted Invoice No. = 168881
Page 25: Extracted Invoice No. = 168882
Page 26: 

#### Working - Using bounding boxes and tolerances to calculate invoice number, split into pages and save each invoice

In [29]:
import fitz  # PyMuPDF
import pandas as pd
import os

def extract_structured_text(pdf_path):
    doc = fitz.open(pdf_path)
    all_blocks = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if "lines" in b:
                for line in b["lines"]:
                    line_text = " ".join([span["text"] for span in line["spans"]])
                    bbox = line["bbox"]
                    bbox = tuple(round(coord) for coord in bbox)
                    all_blocks.append({
                        "page": page_num + 1,
                        "text": line_text,
                        "x0": bbox[0],
                        "y0": bbox[1],
                        "x1": bbox[2],
                        "y1": bbox[3]})
    df = pd.DataFrame(all_blocks)
    return df.sort_values(by=["page", "y0", "x0"])

def find_text_right_of(df, target_text, tolerance=5):
    result = []
    for _, row in df.iterrows():
        if target_text in row["text"]:
            x1_target = row["x1"]
            y0_target = row["y0"]
            y1_target = row["y1"]
            page = row["page"]

            candidates = df[
                (df["page"] == page) &
                (df["x0"] > x1_target) &
                (df["y0"] >= y0_target - tolerance) &
                (df["y1"] <= y1_target + tolerance)]

            if not candidates.empty:
                result.append({
                    "page": page,
                    "invoice_number": candidates.iloc[0]["text"]
                })
    return pd.DataFrame(result)

def split_pdf_by_invoice_number(pdf_path, output_dir="invoices"):
    os.makedirs(output_dir, exist_ok=True)
    df = extract_structured_text(pdf_path)
    invoice_df = find_text_right_of(df, "Invoice No", tolerance=5)

    doc = fitz.open(pdf_path)
    saved_files = []

    for _, row in invoice_df.iterrows():
        page_num = row["page"] - 1  # zero-indexed
        invoice_number = row["invoice_number"]
        new_doc = fitz.open()
        new_doc.insert_pdf(doc, from_page=page_num, to_page=page_num)
        output_path = os.path.join(output_dir, f"invoice_{invoice_number}.pdf")
        new_doc.save(output_path)
        new_doc.close()
        saved_files.append(output_path)

    return saved_files

# Example usage
pdf_path = r"C:\Users\IlaBarshilia\ldiltd.com\M1 Global - M1 Global\Projects\FDOT Invoicing Automation\Samples for PDF Reader\JAX - June CH Billing (extracted) Done.pdf"
split_pdf_by_invoice_number(pdf_path)


['invoices\\invoice_168858.pdf',
 'invoices\\invoice_168859.pdf',
 'invoices\\invoice_168860.pdf',
 'invoices\\invoice_168861.pdf',
 'invoices\\invoice_168862.pdf',
 'invoices\\invoice_168863.pdf',
 'invoices\\invoice_168864.pdf',
 'invoices\\invoice_168865.pdf',
 'invoices\\invoice_168866.pdf',
 'invoices\\invoice_168867.pdf',
 'invoices\\invoice_168868.pdf',
 'invoices\\invoice_168869.pdf',
 'invoices\\invoice_168870.pdf',
 'invoices\\invoice_168871.pdf',
 'invoices\\invoice_168872.pdf',
 'invoices\\invoice_168873.pdf',
 'invoices\\invoice_168874.pdf',
 'invoices\\invoice_168875.pdf',
 'invoices\\invoice_168876.pdf',
 'invoices\\invoice_168877.pdf',
 'invoices\\invoice_168878.pdf',
 'invoices\\invoice_168879.pdf',
 'invoices\\invoice_168880.pdf',
 'invoices\\invoice_168881.pdf',
 'invoices\\invoice_168882.pdf',
 'invoices\\invoice_168883.pdf',
 'invoices\\invoice_168884.pdf',
 'invoices\\invoice_168885.pdf',
 'invoices\\invoice_168886.pdf',
 'invoices\\invoice_168887.pdf',
 'invoices

#### Custom renaming  including CustomerName_Invoice_Date

In [ ]:
import fitz  # PyMuPDF
import pandas as pd
from datetime import datetime
import os
import re


def sanitize_filename(name):
    return re.sub(r'[\\\\/:*?"<>|]', '-', name)


def extract_structured_text(pdf_path):
    doc = fitz.open(pdf_path)
    all_blocks = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if "lines" in b:
                for line in b["lines"]:
                    line_text = " ".join([span["text"] for span in line["spans"]])
                    bbox = line["bbox"]
                    bbox = tuple(round(coord) for coord in bbox)
                    all_blocks.append({
                        "page": page_num + 1,
                        "text": line_text,
                        "x0": bbox[0],
                        "y0": bbox[1],
                        "x1": bbox[2],
                        "y1": bbox[3]
                    })
    df = pd.DataFrame(all_blocks)
    return df.sort_values(by=["page", "y0", "x0"])

def find_text_right_of(df, target_text, tolerance=2):
    result = []
    for _, row in df.iterrows():
        if target_text in row["text"]:
            x1_target = row["x1"]
            y0_target = row["y0"]
            y1_target = row["y1"]
            page = row["page"]

            candidates = df[
                (df["page"] == page) &
                (df["x0"] > x1_target) &
                (df["y0"] >= y0_target - tolerance) &
                (df["y1"] <= y1_target + tolerance)
            ]

            if not candidates.empty:
                result.append({
                    "page": page,
                    "right_text": candidates.iloc[0]["text"]
                })
    return pd.DataFrame(result)

def find_text_left_of(df, target_text, tolerance=2):
    result = []
    for _, row in df.iterrows():
        if target_text in row["text"]:
            x0_target = row["x0"]
            y0_target = row["y0"]
            y1_target = row["y1"]
            page = row["page"]

            candidates = df[
                (df["page"] == page) &
                (df["x1"] < x0_target) &
                (df["y0"] >= y0_target - tolerance) &
                (df["y1"] <= y1_target + tolerance)
            ]

            if not candidates.empty:
                result.append({
                    "page": page,
                    "left_text": candidates.iloc[-1]["text"]
                })
    return pd.DataFrame(result)

def split_pdf_with_custom_naming(pdf_path):
    output_dir = os.path.join(f"split_invoices_of_{pdf_filename}_{datetime.today().strftime('%Y%m%d')}")
    os.makedirs(output_dir, exist_ok=True)
    df = extract_structured_text(pdf_path)

    invoice_df = find_text_right_of(df, "Invoice No", tolerance=2)
    date_df = find_text_right_of(df, "Date", tolerance=2)
    customer_df = find_text_left_of(df, "Job No", tolerance=2)

    doc = fitz.open(pdf_path)
    saved_files = []

    for page_num in range(len(doc)):
        page_index = page_num + 1
        invoice_number = invoice_df[invoice_df["page"] == page_index]["right_text"].values
        date = date_df[date_df["page"] == page_index]["right_text"].values
        customer_name = customer_df[customer_df["page"] == page_index]["left_text"].values

        invoice_number = invoice_number[0] if len(invoice_number) > 0 else "Unknown"
        date = date[0] if len(date) > 0 else "UnknownDate"
        customer_name = customer_name[0] if len(customer_name) > 0 else "UnknownCustomer"

        filename = f"{customer_name} - Inv {invoice_number}, {date}.pdf"
        filename = sanitize_filename(filename)
        output_path = os.path.join(output_dir, filename)

        new_doc = fitz.open()
        new_doc.insert_pdf(doc, from_page=page_num, to_page=page_num)
        new_doc.save(output_path)
        new_doc.close()
        saved_files.append(output_path)

    return saved_files

# Example usage
pdf_path1 = input("Enter the path to where your PDF file is kept that needs to be split: ")
pdf_filename = input("Enter the name of the PDF file in the path shared above that you would like to split: ") + ".pdf"
pdf_path = os.path.join(pdf_path1, pdf_filename)
# pdf_path = r"C:\Users\IlaBarshilia\ldiltd.com\M1 Global - M1 Global\Projects\FDOT Invoicing Automation\Samples for PDF Reader\JAX - June CH Billing (extracted) Done.pdf"
split_pdf_with_custom_naming(pdf_path)


['split_invoices_of_BW - DOT for 07-13-25 (done).pdf_20250808\\Ajax Paving Industries Of Florida LLC - Inv 403295, 07_13_25.pdf',
 'split_invoices_of_BW - DOT for 07-13-25 (done).pdf_20250808\\Ajax Paving Industries Of Florida LLC - Inv 403296, 07_13_25.pdf',
 'split_invoices_of_BW - DOT for 07-13-25 (done).pdf_20250808\\CW Roberts Contracting, Inc. - Inv 403297, 07_13_25.pdf',
 'split_invoices_of_BW - DOT for 07-13-25 (done).pdf_20250808\\CW Roberts Contracting, Inc. - Inv 403298, 07_13_25.pdf',
 'split_invoices_of_BW - DOT for 07-13-25 (done).pdf_20250808\\CW Roberts Contracting, Inc. - Inv 403299, 07_13_25.pdf',
 'split_invoices_of_BW - DOT for 07-13-25 (done).pdf_20250808\\District 3 - Anderson Columbia - Inv 403300, 07_13_25.pdf',
 'split_invoices_of_BW - DOT for 07-13-25 (done).pdf_20250808\\Florida Engineering & Development Corp - Inv 403301, 07_13_25.pdf',
 'split_invoices_of_BW - DOT for 07-13-25 (done).pdf_20250808\\General Asphalt Company - Inv 403302, 07_13_25.pdf',
 'split

In [5]:
pdf_path1 = input("Enter the path to where your PDF file is kept that needs to be split: ")
pdf_filename = input("Enter the name of the PDF file in the path shared above that you would like to split: ")
pdf_path = os.path.join(pdf_path1, pdf_filename)

def split_pdf_with_custom_naming(pdf_path):
    output_dir = os.path.join(pdf_path1, f"split_invoices_of_{pdf_filename}_{datetime.today().strftime('%Y%m%d')}")
    os.makedirs(output_dir, exist_ok=True)

    df = extract_structured_text(os.path.join(pdf_path1, pdf_filename + ".pdf"))

    invoice_df = find_text_right_of(df, "Invoice No", tolerance=2)
    date_df = find_text_right_of(df, "Date", tolerance=2)
    customer_df = find_text_left_of(df, "Job No", tolerance=2)

    doc = fitz.open(os.path.join(pdf_path1, pdf_filename + ".pdf"))
    saved_files = []

    current_invoice_number = "Unknown"
    current_date = "UnknownDate"
    current_customer_name = "UnknownCustomer"
    current_pages = []

    for page_num in range(len(doc)):
        page_index = page_num + 1
        invoice_number = invoice_df[invoice_df["page"] == page_index]["right_text"].values
        date = date_df[date_df["page"] == page_index]["right_text"].values
        customer_name = customer_df[customer_df["page"] == page_index]["left_text"].values

        if len(invoice_number) > 0:
            # Save previous invoice if pages exist
            if current_pages:
                filename = f"{current_customer_name} - Inv {current_invoice_number}, {current_date}.pdf"
                filename = sanitize_filename(filename)
                output_path = os.path.join(output_dir, filename)

                new_doc = fitz.open()
                for p in current_pages:
                    new_doc.insert_pdf(doc, from_page=p, to_page=p)
                new_doc.save(output_path)
                new_doc.close()
                saved_files.append(output_path)

            # Start new invoice group
            current_invoice_number = invoice_number[0]
            current_date = date[0] if len(date) > 0 else "UnknownDate"
            current_customer_name = customer_name[0] if len(customer_name) > 0 else "UnknownCustomer"
            current_pages = [page_num]
        else:
            # Continue current invoice group
            current_pages.append(page_num)

    # Save the last invoice group
    if current_pages:
        filename = f"{current_customer_name} - Inv {current_invoice_number}, {current_date}.pdf"
        filename = sanitize_filename(filename)
        output_path = os.path.join(output_dir, filename)

        new_doc = fitz.open()
        for p in current_pages:
            new_doc.insert_pdf(doc, from_page=p, to_page=p)
        new_doc.save(output_path)
        new_doc.close()
        saved_files.append(output_path)

    return saved_files

split_pdf_with_custom_naming(pdf_path)

['C:\\Users\\IlaBarshilia\\ldiltd.com\\M1 Global - M1 Global\\Projects\\FDOT Invoicing Automation\\Samples for PDF Reader\\split_invoices_of_JAX - June CH Billing (extracted) Done_20250808\\4-K Construction - Inv 168858, 06_30_25.pdf',
 'C:\\Users\\IlaBarshilia\\ldiltd.com\\M1 Global - M1 Global\\Projects\\FDOT Invoicing Automation\\Samples for PDF Reader\\split_invoices_of_JAX - June CH Billing (extracted) Done_20250808\\4-K Construction - Inv 168859, 06_30_25.pdf',
 'C:\\Users\\IlaBarshilia\\ldiltd.com\\M1 Global - M1 Global\\Projects\\FDOT Invoicing Automation\\Samples for PDF Reader\\split_invoices_of_JAX - June CH Billing (extracted) Done_20250808\\4-K Construction - Inv 168860, 06_30_25.pdf',
 'C:\\Users\\IlaBarshilia\\ldiltd.com\\M1 Global - M1 Global\\Projects\\FDOT Invoicing Automation\\Samples for PDF Reader\\split_invoices_of_JAX - June CH Billing (extracted) Done_20250808\\A.J. Johns - Inv 168861, 06_30_25.pdf',
 'C:\\Users\\IlaBarshilia\\ldiltd.com\\M1 Global - M1 Global\\

#### Working - Extracting and Organising text as per PDF

In [1]:
import fitz  # PyMuPDF
import pandas as pd

def extract_structured_text(pdf_path):
    doc = fitz.open(pdf_path)
    all_blocks = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if "lines" in b:
                for line in b["lines"]:
                    for span in line["spans"]:
                        bbox = span["bbox"]
                        bbox = tuple(round(coord) for coord in bbox)
                        all_blocks.append({
                            "page": page_num + 1,
                            "text": span["text"],
                            "x0": bbox[0],
                            "y0": bbox[1],
                            "x1": bbox[2],
                            "y1": bbox[3]
                        })

    df = pd.DataFrame(all_blocks)
    return df.sort_values(by=["page", "y0", "x0"])

# Example usage
# pdf_path = r"C:\Users\IlaBarshilia\ldiltd.com\M1 Global - M1 Global\Projects\FDOT Invoicing Automation\Samples for PDF Reader\JAX - June CH Billing (extracted) Done.pdf"
pdf_path = r"C:\Users\IlaBarshilia\ldiltd.com\M1 Global - M1 Global\Projects\FDOT Invoicing Automation\Samples for PDF Reader\split_invoices_of_BW - DOT for 07-13-25 (done)_2025-Aug-08\CW Roberts Contracting, Inc. - Inv 403297, 07-13-25.pdf"
df = extract_structured_text(pdf_path)

def group_by_dynamic_columns(df, y_tolerance=2, x_bin_width=20):
    from collections import defaultdict

    df = df.sort_values(by=["page", "y0", "x0"]).reset_index(drop=True)
    structured_rows = []

    for page in df["page"].unique():
        page_df = df[df["page"] == page].copy()
        rows = []

        for _, row in page_df.iterrows():
            y0 = row["y0"]
            matched = False

            # Try to place in an existing row
            for r in rows:
                if abs(r["y0"] - y0) <= y_tolerance:
                    r["cells"].append(row)
                    matched = True
                    break

            # If no match, start a new row
            if not matched:
                rows.append({"y0": y0, "cells": [row]})

        # Now sort each row's cells into dynamic x-bins
        for r in rows:
            bins = defaultdict(list)
            for cell in r["cells"]:
                bin_index = int(cell["x0"] // x_bin_width)
                bins[bin_index].append(cell["text"])

            # Sort bins by x-position and join text in each bin
            sorted_bins = [bins[i] for i in sorted(bins)]
            structured_row = [" ".join(bin) for bin in sorted_bins]
            # Ensure bins are aligned by filling missing indices with empty strings -use this if all bins are needed if None values
            # max_bin = max(bins.keys())
            # structured_row = [" ".join(bins[i]) if i in bins else "" for i in range(max_bin + 1)]

            structured_rows.append(structured_row)

    return pd.DataFrame(structured_rows)

df2 = group_by_dynamic_columns(df)

RuntimeError: Directory 'static/' does not exist

In [ ]:
import re
start_index = df2[df2.apply(lambda row: row.astype(str).str.contains("Invoice No").any(), axis=1)].index.min()
# Slice the DataFrame from that index onward
df3 = df2.loc[start_index:].reset_index(drop=True)
df3 = df3.drop(df3.index[2:8]).reset_index(drop=True)

# List of target values to check
right_target_values = ["Invoice No", "Date", "ACME Job", "REF:"]
# Store results
right_values = []

# Loop through each row
for index, row in df3.iterrows():
    for col_index, value in enumerate(row):
        if str(value).strip() in right_target_values:
            # Check if there's a column to the right
            if col_index + 1 < len(row):
                right_value = row.iloc[col_index + 1]
                right_values.append((index, value, right_value))


processed_right = []
for idx, label, value in right_values:
    if label == "REF:":
        new_label = "FDOT_Contract_ID"
        words = re.findall(r'\b\w{5}\b', value)
        # Filter words based on conditions:
        # - First character is a letter
        # - Remaining 4 characters contain either:
        #   * 3 digits and 1 letter
        #   * 4 digits
        valid_words = []
        for word in words:
            if word[0].isalpha():
                rest = word[1:]
                digits = sum(c.isdigit() for c in rest)
                letters = sum(c.isalpha() for c in rest)
                if (digits == 3 and letters == 1) or (digits == 4 and letters == 0):
                    valid_words.append(word)

        # Join results with comma
        new_value = ", ".join(valid_words)
        # new_value = value.split(" ")[0]  # Extract 'H1210' before first space
        processed_right.append((idx, new_label, new_value))
    else:
        processed_right.append((idx, label, value))

# Display results
for idx, target, right in processed_right:
    print(f" '{target}' is {right}")
# List of target values to check
left_target_value = ["Job No"]
# Store results
left_values = []
# Loop through each row
for index, row in df3.iterrows():
    for col_index, value in enumerate(row):
        if str(value).strip() in left_target_value:
            # Check if there's a column to the left
            if col_index - 1 >= 0:
                left_value = row.iloc[col_index - 1]
                left_values.append((index, value, left_value))

processed_left = []
for idx, label, value in left_values:
    if label == "Job No":
        new_label = "Customer Name"
        processed_left.append((idx, new_label, value))
    else:
        processed_left.append((idx, label, value))

start_index = df3[df3.apply(lambda row: row.astype(str).str.contains("Item").any(), axis=1)].index.min()
# Slice the DataFrame from that index onward
df4 = df3.loc[start_index:].reset_index(drop=True)

df4.columns = df4.iloc[0]
df4 = df4[1:].reset_index(drop=True)

# Column to check
column_name = "Item"

# List of unwanted values
unwanted_values = ["Rentals:", "Tax","Progress Bill", "Total Taxes", "Invoice Total"]

# Filter out rows where the column contains any unwanted value
df4 = df4[~df4["Item"].astype(str).isin(unwanted_values)].reset_index(drop=True)
df4 = df4[(df4["Equipment and Labor"]!= "Subtotal")]

# Columns to shift
columns_to_shift = ['To', 'Hrs/Days', 'Qty', 'Price', 'Amount']

# Identify the trigger index
matches = df4.index[df4['Item'].str.strip() == 'Services:']
if len(matches) > 0:
    trigger_index = matches[0]
    # Perform the shift for all rows below the trigger
    for i in range(trigger_index + 1, len(df4)):
        for j in range(len(columns_to_shift) - 1, 0, -1):
            df4.at[i, columns_to_shift[j]] = df4.at[i, columns_to_shift[j - 1]]
        df4.at[i, columns_to_shift[0]] = None  # Clear the first column after shifting
else:
    print("No 'Services:' found in Item column")

df4 = df4[df4["Item"]!= "Services:"]
df4


 'Invoice No' is 403297
 'Date' is 07/13/25
 'ACME Job' is 21532
 'FDOT_Contract_ID' is T1878


,Item,Equipment and Labor,From,To,Hrs/Days,Qty,Price,Amount
0,CBW-K,Concrete Barrier Wall K Wall,11/18/24,06/23/25,218,21,$0.00,$0.00
1,CBW-K,Concrete Barrier Wall K Wall,11/18/24,07/13/25,238,110,$0.00,$0.00
2,LOWPRO,Low Pro Wall,11/18/24,06/23/25,218,74,$0.00,$0.00
4,102-71-26,"Temp Barrier Wall, Relocate Free",06/23/25,None,1,325,$14.00,"$4,550.00"
5,Standing (LF),None,None,None,None,None,None,None
6,102-71-26,"Temp Barrier Wall, Relocate Free",06/24/25,None,1,1000,$14.00,"$14,000.00"
7,Standing (LF),None,None,None,None,None,None,None


In [30]:
FDOT_output = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\FDOT_Output_Data_2025-Aug-26.xlsx")
FDOT_output=FDOT_output[FDOT_output["Contract"]==processed_right[3][2]]
FDOT_output

,Contract,Vendor Name,EstimateNumber,EstimateEnd Date,Project,Unit,Item Code,Previous,This Est.,To-Date,Item Description
2521,T1878,"C.W. ROBERTS CONTRACTING, INC.",2,2025-08-17,441977-1-52-01,LF,0102 71 13,886.000,0.000,886.000,"TEMPORARY BARRIER, F&I, LOW PROFILE, CONCRETE"
2522,T1878,"C.W. ROBERTS CONTRACTING, INC.",4,2025-08-17,441977-1-52-01,LF,0102 71 15,1663.000,0.000,1663.000,"TEMPORARY BARRIER, F&I, ANCHORED"
2523,T1878,"C.W. ROBERTS CONTRACTING, INC.",4,2025-08-17,441977-1-52-01,ED,0102150 1,120.000,0.000,120.000,"PORTABLE REGULATORY, SIGN"
2524,T1878,"C.W. ROBERTS CONTRACTING, INC.",4,2025-08-17,441977-1-52-01,ED,0102150 2,176.000,0.000,176.000,RADAR SPEED DISPLAY UNIT
2525,T1878,"C.W. ROBERTS CONTRACTING, INC.",5,2025-08-17,441977-1-52-01,LS,0101 1,1.000,0.000,1.000,MOBILIZATION-44197715201
2526,T1878,"C.W. ROBERTS CONTRACTING, INC.",11,2025-08-17,441977-1-52-01,HR,0102 14,139.500,0.000,139.500,TRAFFIC CONTROL OFFICER
2527,T1878,"C.W. ROBERTS CONTRACTING, INC.",11,2025-08-17,441977-1-52-01,LF,0102 71 26,1325.000,0.000,1325.000,"TEMPORARY BARRIER, RELOCATE, FREE STANDING"
2528,T1878,"C.W. ROBERTS CONTRACTING, INC.",11,2025-08-17,441977-1-52-01,LO,0102 89 1,3.000,0.000,3.000,"TEMPORARY CRASH CUSHION, REDIRECTIVE OPTION"
2529,T1878,"C.W. ROBERTS CONTRACTING, INC.",11,2025-08-17,441977-1-52-01,GM,0102913 21,0.273,0.000,0.273,"REMOVABLE TAPE, WHITE, SOLID 6"""
2530,T1878,"C.W. ROBERTS CONTRACTING, INC.",11,2025-08-17,441977-1-52-01,GM,0102913 31,0.398,0.000,0.398,"REMOVABLE TAPE, YELLOW, SOLID, 6"""
